In [7]:
import json
import urllib.error
import urllib.request


def fetch_ai_status(url: str = "http://localhost:7071/api/ai/status", timeout: int = 5) -> dict:
    try:
        with urllib.request.urlopen(url, timeout=timeout) as response:
            payload = response.read().decode("utf-8")
            return json.loads(payload)
    except urllib.error.HTTPError as exc:
        return {"ok": False, "error": f"HTTP {exc.code}", "detail": str(exc)}
    except urllib.error.URLError as exc:
        return {"ok": False, "error": "Connection failed", "detail": str(exc.reason)}
    except Exception as exc:
        return {"ok": False, "error": "Unexpected error", "detail": str(exc)}


status = fetch_ai_status()
print(json.dumps(status, indent=2, sort_keys=True))

{
  "detail": "[Errno 111] Connection refused",
  "error": "Connection failed",
  "ok": false
}


In [8]:
def summarize_ai_status(status: dict) -> dict:
    provider = status.get("active_provider") or status.get(
        "provider") or "unknown"
    env = status.get("environment") or status.get("env") or {}
    sql = status.get("sql") or {}
    cosmos = status.get("cosmos") or {}

    sql_pool = sql.get("pool") if isinstance(sql, dict) else None
    pool_used = None
    pool_size = None
    if isinstance(sql_pool, dict):
        pool_used = sql_pool.get("used")
        pool_size = sql_pool.get("size")

    warnings = []
    if status.get("ok") is False:
        warnings.append(status.get(
            "error", "status endpoint returned ok=False"))

    if isinstance(pool_used, int) and isinstance(pool_size, int) and pool_size > 0:
        saturation = pool_used / pool_size
        if saturation >= 0.8:
            warnings.append(f"SQL pool saturation high ({saturation:.0%})")

    if isinstance(cosmos, dict) and cosmos.get("enabled") and not cosmos.get("healthy", True):
        warnings.append("Cosmos is enabled but unhealthy")

    required_azure = [
        "AZURE_OPENAI_API_KEY",
        "AZURE_OPENAI_ENDPOINT",
        "AZURE_OPENAI_DEPLOYMENT",
        "AZURE_OPENAI_API_VERSION",
    ]
    missing_azure = [k for k in required_azure if not env.get(
        k)] if isinstance(env, dict) else required_azure

    return {
        "provider": provider,
        "overall_ok": status.get("ok", True),
        "missing_azure_env": missing_azure,
        "sql_pool_used": pool_used,
        "sql_pool_size": pool_size,
        "cosmos_enabled": cosmos.get("enabled") if isinstance(cosmos, dict) else False,
        "cosmos_healthy": cosmos.get("healthy") if isinstance(cosmos, dict) else None,
        "warnings": warnings,
    }


summary = summarize_ai_status(status)
print(json.dumps(summary, indent=2, sort_keys=True))

{
  "cosmos_enabled": null,
  "cosmos_healthy": null,
  "missing_azure_env": [
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_DEPLOYMENT",
    "AZURE_OPENAI_API_VERSION"
  ],
  "overall_ok": false,
  "provider": "unknown",
  "sql_pool_size": null,
  "sql_pool_used": null,
  "warnings": [
    "Connection failed"
  ]
}


In [9]:
def build_readiness_report(summary: dict) -> dict:
    checks = {
        "status_ok": bool(summary.get("overall_ok", False)),
        "provider_detected": summary.get("provider") not in (None, "", "unknown"),
        "azure_env_complete": len(summary.get("missing_azure_env", [])) == 0,
        "sql_pool_configured": summary.get("sql_pool_size") is not None,
        "cosmos_if_enabled_is_healthy": (
            not summary.get("cosmos_enabled", False)
            or bool(summary.get("cosmos_healthy", False))
        ),
    }

    recommendations = []
    if not checks["status_ok"]:
        recommendations.append("Start Azure Functions host: func host start")
    if not checks["provider_detected"]:
        recommendations.append("Set provider-related environment variables")
    if not checks["azure_env_complete"]:
        missing = ", ".join(summary.get("missing_azure_env", []))
        recommendations.append(
            f"Populate missing Azure OpenAI vars: {missing}")
    if summary.get("sql_pool_size") and summary.get("sql_pool_used") is not None:
        used = summary["sql_pool_used"]
        size = summary["sql_pool_size"]
        if isinstance(used, int) and isinstance(size, int) and size > 0 and (used / size) >= 0.8:
            recommendations.append(
                "Reduce DB load or increase QAI_SQL_POOL_SIZE")
    if summary.get("cosmos_enabled") and not summary.get("cosmos_healthy"):
        recommendations.append(
            "Check Cosmos endpoint/key/database/container settings")

    score = sum(1 for ok in checks.values() if ok)
    return {
        "readiness_score": f"{score}/{len(checks)}",
        "checks": checks,
        "warnings": summary.get("warnings", []),
        "recommendations": recommendations,
    }


report = build_readiness_report(summary)
print(json.dumps(report, indent=2, sort_keys=True))

{
  "checks": {
    "azure_env_complete": false,
    "cosmos_if_enabled_is_healthy": true,
    "provider_detected": false,
    "sql_pool_configured": false,
    "status_ok": false
  },
  "readiness_score": "1/5",
  "recommendations": [
    "Start Azure Functions host: func host start",
    "Set provider-related environment variables",
    "Populate missing Azure OpenAI vars: AZURE_OPENAI_API_KEY, AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_DEPLOYMENT, AZURE_OPENAI_API_VERSION"
  ],
  "warnings": [
    "Connection failed"
  ]
}


In [10]:
final_status = "READY" if report["readiness_score"].startswith(
    "5/") else "ACTION_NEEDED"

result = {
    "final_status": final_status,
    "readiness_score": report.get("readiness_score"),
    "warning_count": len(report.get("warnings", [])),
    "recommendation_count": len(report.get("recommendations", [])),
}

print(json.dumps(result, indent=2, sort_keys=True))

if report.get("recommendations"):
    print("\nTop recommendation:")
    print(f"- {report['recommendations'][0]}")

{
  "final_status": "ACTION_NEEDED",
  "readiness_score": "1/5",
  "recommendation_count": 3,
  "warning_count": 1
}

Top recommendation:
- Start Azure Functions host: func host start


In [11]:
import json
import os
import datetime

out_dir = os.path.join("data_out", "ai_status_reports")
os.makedirs(out_dir, exist_ok=True)

timestamp = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
output_path = os.path.join(out_dir, f"ai_status_report_{timestamp}.json")

payload = {
    "generated_at_utc": timestamp,
    "status": status,
    "summary": summary,
    "report": report,
    "result": result,
}

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, sort_keys=True)

print(f"Saved report: {output_path}")

Saved report: data_out/ai_status_reports/ai_status_report_20260329T044427Z.json


In [12]:
import glob
import json
import os
from typing import Optional


def _extract_readiness_numerator(score: Optional[str]) -> Optional[int]:
    if not score or "/" not in score:
        return None
    try:
        return int(str(score).split("/", 1)[0])
    except Exception:
        return None


def load_previous_report(reports_dir: str, current_path: str) -> Optional[dict]:
    files = sorted(glob.glob(os.path.join(
        reports_dir, "ai_status_report_*.json")))
    prev_candidates = [p for p in files if os.path.abspath(
        p) != os.path.abspath(current_path)]
    if not prev_candidates:
        return None
    prev_path = prev_candidates[-1]
    with open(prev_path, "r", encoding="utf-8") as f:
        return {"path": prev_path, "data": json.load(f)}


previous = load_previous_report(out_dir, output_path)
comparison = {
    "current_report": output_path,
    "previous_report": previous["path"] if previous else None,
    "delta": {},
}

if previous:
    prev_data = previous["data"]

    current_score = report.get("readiness_score")
    prev_score = (prev_data.get("report") or {}).get("readiness_score")

    current_score_num = _extract_readiness_numerator(current_score)
    prev_score_num = _extract_readiness_numerator(prev_score)

    current_warnings = len(report.get("warnings", []))
    prev_warnings = len((prev_data.get("report") or {}).get("warnings", []))

    current_recs = len(report.get("recommendations", []))
    prev_recs = len((prev_data.get("report") or {}).get("recommendations", []))

    comparison["delta"] = {
        "readiness_score_current": current_score,
        "readiness_score_previous": prev_score,
        "readiness_points_change": (
            None if current_score_num is None or prev_score_num is None else current_score_num - prev_score_num
        ),
        "warning_count_change": current_warnings - prev_warnings,
        "recommendation_count_change": current_recs - prev_recs,
        "status_changed": result.get("final_status") != ((prev_data.get("result") or {}).get("final_status")),
        "status_current": result.get("final_status"),
        "status_previous": (prev_data.get("result") or {}).get("final_status"),
    }

print(json.dumps(comparison, indent=2, sort_keys=True))

{
  "current_report": "data_out/ai_status_reports/ai_status_report_20260329T044427Z.json",
  "delta": {
    "readiness_points_change": 0,
    "readiness_score_current": "1/5",
    "readiness_score_previous": "1/5",
    "recommendation_count_change": 0,
    "status_changed": false,
    "status_current": "ACTION_NEEDED",
    "status_previous": "ACTION_NEEDED",
    "warning_count_change": 0
  },
  "previous_report": "data_out/ai_status_reports/ai_status_report_20260329T043853Z.json"
}


In [13]:
import datetime
import glob
import json
import os
from typing import Any, Dict, Optional


def run_ai_status_pipeline(
    url: str = "http://localhost:7071/api/ai/status",
    timeout: int = 5,
    reports_dir: str = os.path.join("data_out", "ai_status_reports"),
    save_report: bool = True,
    compare_previous: bool = True,
) -> Dict[str, Any]:
    status = fetch_ai_status(url=url, timeout=timeout)
    summary = summarize_ai_status(status)
    report = build_readiness_report(summary)

    final_status = "READY" if str(report.get(
        "readiness_score", "")).startswith("5/") else "ACTION_NEEDED"
    result = {
        "final_status": final_status,
        "readiness_score": report.get("readiness_score"),
        "warning_count": len(report.get("warnings", [])),
        "recommendation_count": len(report.get("recommendations", [])),
    }

    output_path: Optional[str] = None
    if save_report:
        os.makedirs(reports_dir, exist_ok=True)
        timestamp = datetime.datetime.now(
            datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        output_path = os.path.join(
            reports_dir, f"ai_status_report_{timestamp}.json")
        payload = {
            "generated_at_utc": timestamp,
            "status": status,
            "summary": summary,
            "report": report,
            "result": result,
        }
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2, sort_keys=True)

    comparison: Dict[str, Any] = {
        "current_report": output_path,
        "previous_report": None,
        "delta": {},
    }

    if compare_previous and output_path:
        files = sorted(glob.glob(os.path.join(
            reports_dir, "ai_status_report_*.json")))
        prev_candidates = [p for p in files if os.path.abspath(
            p) != os.path.abspath(output_path)]
        if prev_candidates:
            prev_path = prev_candidates[-1]
            with open(prev_path, "r", encoding="utf-8") as f:
                prev_data = json.load(f)

            def _score_num(s: Optional[str]) -> Optional[int]:
                if not s or "/" not in str(s):
                    return None
                try:
                    return int(str(s).split("/", 1)[0])
                except Exception:
                    return None

            current_score = report.get("readiness_score")
            prev_score = (prev_data.get("report") or {}).get("readiness_score")
            current_score_num = _score_num(current_score)
            prev_score_num = _score_num(prev_score)

            current_warnings = len(report.get("warnings", []))
            prev_warnings = len(
                (prev_data.get("report") or {}).get("warnings", []))
            current_recs = len(report.get("recommendations", []))
            prev_recs = len((prev_data.get("report") or {}
                             ).get("recommendations", []))

            comparison = {
                "current_report": output_path,
                "previous_report": prev_path,
                "delta": {
                    "readiness_score_current": current_score,
                    "readiness_score_previous": prev_score,
                    "readiness_points_change": (
                        None
                        if current_score_num is None or prev_score_num is None
                        else current_score_num - prev_score_num
                    ),
                    "warning_count_change": current_warnings - prev_warnings,
                    "recommendation_count_change": current_recs - prev_recs,
                    "status_changed": result.get("final_status") != ((prev_data.get("result") or {}).get("final_status")),
                    "status_current": result.get("final_status"),
                    "status_previous": (prev_data.get("result") or {}).get("final_status"),
                },
            }

    return {
        "status": status,
        "summary": summary,
        "report": report,
        "result": result,
        "comparison": comparison,
    }


pipeline_output = run_ai_status_pipeline()
print(json.dumps(pipeline_output["result"], indent=2, sort_keys=True))
print(json.dumps(pipeline_output["comparison"], indent=2, sort_keys=True))

{
  "final_status": "ACTION_NEEDED",
  "readiness_score": "1/5",
  "recommendation_count": 3,
  "warning_count": 1
}
{
  "current_report": "data_out/ai_status_reports/ai_status_report_20260329T044427Z.json",
  "delta": {
    "readiness_points_change": 0,
    "readiness_score_current": "1/5",
    "readiness_score_previous": "1/5",
    "recommendation_count_change": 0,
    "status_changed": false,
    "status_current": "ACTION_NEEDED",
    "status_previous": "ACTION_NEEDED",
    "warning_count_change": 0
  },
  "previous_report": "data_out/ai_status_reports/ai_status_report_20260329T043853Z.json"
}


In [14]:
import glob
import json
import os
from typing import Any, Dict, List, Optional


def _score_to_float(score: Optional[str]) -> Optional[float]:
    if not score or "/" not in str(score):
        return None
    try:
        num, den = str(score).split("/", 1)
        num_i = int(num)
        den_i = int(den)
        if den_i <= 0:
            return None
        return num_i / den_i
    except Exception:
        return None


def analyze_report_history(
    reports_dir: str = os.path.join("data_out", "ai_status_reports"),
    limit: int = 20,
) -> Dict[str, Any]:
    files = sorted(glob.glob(os.path.join(
        reports_dir, "ai_status_report_*.json")))[-limit:]
    history: List[Dict[str, Any]] = []

    for path in files:
        with open(path, "r", encoding="utf-8") as f:
            payload = json.load(f)
        report = payload.get("report") or {}
        result = payload.get("result") or {}
        score = report.get("readiness_score")
        readiness = _score_to_float(score)
        history.append(
            {
                "path": path,
                "generated_at_utc": payload.get("generated_at_utc"),
                "score": score,
                "readiness": readiness,
                "status": result.get("final_status"),
                "warnings": len(report.get("warnings") or []),
                "recommendations": len(report.get("recommendations") or []),
            }
        )

    degradations: List[Dict[str, Any]] = []
    for i in range(1, len(history)):
        prev = history[i - 1]
        cur = history[i]
        prev_val = prev.get("readiness")
        cur_val = cur.get("readiness")
        if isinstance(prev_val, float) and isinstance(cur_val, float):
            delta = cur_val - prev_val
            if delta <= -0.05:
                degradations.append(
                    {
                        "from": prev.get("generated_at_utc"),
                        "to": cur.get("generated_at_utc"),
                        "delta": round(delta, 4),
                    }
                )

    latest = history[-1] if history else {}
    avg_readiness = None
    readiness_values = [h["readiness"]
                        for h in history if isinstance(h.get("readiness"), float)]
    if readiness_values:
        avg_readiness = round(sum(readiness_values) / len(readiness_values), 4)

    return {
        "report_count": len(history),
        "latest": latest,
        "avg_readiness": avg_readiness,
        "degradation_alerts": degradations,
        "history": history,
    }


history_summary = analyze_report_history()
print(json.dumps({
    "report_count": history_summary["report_count"],
    "latest": history_summary["latest"],
    "avg_readiness": history_summary["avg_readiness"],
    "degradation_alert_count": len(history_summary["degradation_alerts"]),
}, indent=2, sort_keys=True))

if history_summary["degradation_alerts"]:
    print("\nDegradation alerts (>5% readiness drop):")
    for alert in history_summary["degradation_alerts"]:
        print(f"- {alert['from']} -> {alert['to']}: {alert['delta']:+.2%}")

{
  "avg_readiness": 0.2,
  "degradation_alert_count": 0,
  "latest": {
    "generated_at_utc": "20260329T044427Z",
    "path": "data_out/ai_status_reports/ai_status_report_20260329T044427Z.json",
    "readiness": 0.2,
    "recommendations": 3,
    "score": "1/5",
    "status": "ACTION_NEEDED",
    "warnings": 1
  },
  "report_count": 2
}


In [15]:
import json
import os
from typing import Any, Dict


def run_and_print_ai_status_snapshot(
    url: str | None = None,
    timeout: int | None = None,
    save_report: bool = True,
    compare_previous: bool = True,
    include_history: bool = True,
    history_limit: int = 20,
) -> Dict[str, Any]:
    effective_url = url or os.getenv(
        "AI_STATUS_URL", "http://localhost:7071/api/ai/status")
    effective_timeout = timeout if timeout is not None else int(
        os.getenv("AI_STATUS_TIMEOUT", "5"))

    output = run_ai_status_pipeline(
        url=effective_url,
        timeout=effective_timeout,
        save_report=save_report,
        compare_previous=compare_previous,
    )

    view: Dict[str, Any] = {
        "url": effective_url,
        "timeout": effective_timeout,
        "result": output.get("result", {}),
        "comparison": output.get("comparison", {}),
    }

    if include_history:
        history = analyze_report_history(limit=history_limit)
        view["history_summary"] = {
            "report_count": history.get("report_count", 0),
            "avg_readiness": history.get("avg_readiness"),
            "degradation_alert_count": len(history.get("degradation_alerts", [])),
            "latest": history.get("latest", {}),
        }

    print(json.dumps(view, indent=2, sort_keys=True))
    return output


snapshot = run_and_print_ai_status_snapshot()

{
  "comparison": {
    "current_report": "data_out/ai_status_reports/ai_status_report_20260329T044427Z.json",
    "delta": {
      "readiness_points_change": 0,
      "readiness_score_current": "1/5",
      "readiness_score_previous": "1/5",
      "recommendation_count_change": 0,
      "status_changed": false,
      "status_current": "ACTION_NEEDED",
      "status_previous": "ACTION_NEEDED",
      "warning_count_change": 0
    },
    "previous_report": "data_out/ai_status_reports/ai_status_report_20260329T043853Z.json"
  },
  "history_summary": {
    "avg_readiness": 0.2,
    "degradation_alert_count": 0,
    "latest": {
      "generated_at_utc": "20260329T044427Z",
      "path": "data_out/ai_status_reports/ai_status_report_20260329T044427Z.json",
      "readiness": 0.2,
      "recommendations": 3,
      "score": "1/5",
      "status": "ACTION_NEEDED",
      "warnings": 1
    },
    "report_count": 2
  },
  "result": {
    "final_status": "ACTION_NEEDED",
    "readiness_score": "1/5"